In [ ]:
!pip install fastapi nest-asyncio uvicorn

In [ ]:
yaml_config =  """
               model_path: "MoritzLaurer/mDeBERTa-v3-base-mnli-xnli"
               """
with open("config.yaml", "w", encoding="utf-8") as f:
    f.write(yaml_config)

In [ ]:
!pip install transformers torch

In [ ]:
from transformers import pipeline
classifier = pipeline("zero-shot-classification",
                      model="facebook/bart-large-mnli")
sequence_to_classify = "I just passed my exam with a 10"
candidate_labels = [
    "stressed",
    "sad",
    "angry",
    "tired",
    "unfocused",
    "happy"
]
output = classifier(sequence_to_classify, candidate_labels)
print(output['labels'][0])

Loading weights:   0%|          | 0/515 [00:00<?, ?it/s]

happy


In [ ]:
import torch
from omegaconf import OmegaConf
from transformers import pipeline, DistilBertTokenizer, DistilBertForSequenceClassification


class MoodClassification:
    def __init__(self, config_path=None):
        self.classifier = pipeline(
            "zero-shot-classification",
            model="facebook/bart-large-mnli"
        )
        self.labels = [
            "stressed",
            "sad",
            "angry",
            "tired",
            "unfocused",
            "happy"
        ]

    def __call__(self, text):
        result = self.classifier(text, self.labels)
        return result['labels'][0]

In [ ]:
from fastapi import FastAPI, HTTPException
import uvicorn
import nest_asyncio
from threading import Thread

nest_asyncio.apply()
app = FastAPI()

mood_classifier = MoodClassification('config.yaml')
@app.get('/')
def read_root():
  return {
          "Name": "Mood Analysis System",
          "Model": "facebook/bart-large-mnli"
         }

@app.get('/health')
def health():
    # Kiểm tra xem model đã được khởi tạo chưa
    model_status = "ready" if mood_classifier is not None else "loading"

    # Kiểm tra thiết bị phần cứng đang sử dụng
    device = "cuda (GPU)" if torch.cuda.is_available() else "cpu"
    # Tính thời gian server đã chạy (Uptime)


    return {
        "status": "healthy",
        "details": {
            "model_name": "facebook/bart-large-mnli",
            "model_status": model_status,
            "device": device,
            "memory_usage": "stable"
        }
    }

@app.post('/CLImood')
def post_mood(message: str):
  if not message or len(message.strip()) == 0:
        raise HTTPException(status_code=400, detail="Message không được để trống")
  try:
    mood = mood_classifier(message)
    return {
        "status": "ok",
        "data": {
            "mood": mood
        }
    }
  except Exception as e:
    raise HTTPException(status_code=500, detail=str(e))

def run_server():
    uvicorn.run(app, host="0.0.0.0", port=8000)

# Chạy server trong thread riêng
thread = Thread(target=run_server, daemon=True)
thread.start()
print("Server started on http://0.0.0.0:8000")

/usr/local/lib/python3.12/dist-packages/huggingface_hub/utils/_auth.py:94: UserWarning: 
The secret `HF_TOKEN` does not exist in your Colab secrets.
To authenticate with the Hugging Face Hub, create a token in your settings tab (https://huggingface.co/settings/tokens), set it as secret in your Google Colab and restart your session.
You will be able to reuse this secret in all of your notebooks.
Please note that authentication is recommended but still optional to access public models or datasets.
  warnings.warn(


Loading weights:   0%|          | 0/515 [00:00<?, ?it/s]

Server started on http://0.0.0.0:8000


In [ ]:
import requests
API_URL = "http://127.0.0.1:8000/CLImood"
params = {"message": "I just broke up with my girl friend"}
response = requests.get(API_URL, params=params)
print(response.json())

INFO:     127.0.0.1:38160 - "GET /CLImood?message=I+just+broke+up+with+my+girl+friend HTTP/1.1" 200 OK
{'Mood': 'sad'}


In [ ]:
import requests

API_URL = "http://einjc-34-172-120-144.run.pinggy-free.link/CLImood"
params = {"message": "I just broke up with my girl friend"}
response = requests.post(API_URL, params=params)
ROOT_URL = "http://einjc-34-172-120-144.run.pinggy-free.link/"
response2 = requests.get(ROOT_URL)
HEALTH_URL = "http://einjc-34-172-120-144.run.pinggy-free.link/health"
response3 = requests.get(HEALTH_URL)
print(response.json())
print(response2.json())
print(response3.json())

INFO:     34.172.120.144:0 - "POST /CLImood?message=I+just+broke+up+with+my+girl+friend HTTP/1.1" 200 OK
INFO:     34.172.120.144:0 - "GET / HTTP/1.1" 200 OK
INFO:     34.172.120.144:0 - "GET /health HTTP/1.1" 200 OK
{'status': 'ok', 'data': {'mood': 'sad'}}
{'Name': 'Mood Analysis System', 'Model': 'facebook/bart-large-mnli'}
{'status': 'healthy', 'details': {'model_name': 'facebook/bart-large-mnli', 'model_status': 'ready', 'device': 'cpu', 'memory_usage': 'stable'}}


In [ ]:
!fuser -k 8000/tcp